# 10-Fold Cross-Validation for XGBoost Model
This notebook performs 10-fold cross-validation to evaluate an XGBoost model predicting negative thrombophilia test results from rule-based antecedents.

In [ ]:

import pandas as pd
import numpy as np
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score


In [ ]:

# Load data
file_path = "reglas.xlsx"  # Adjust as needed
df = pd.read_excel(file_path, sheet_name="Original")

# Filter and label
df = df[df['consequents'].astype(str).str.contains("ana_dura")]
df = df[df['lift'] > 1]
df['label'] = df['consequents'].astype(str).apply(lambda x: 1 if "ana_dura:Buscada negativo" in x else 0)

# Parse antecedents
def parse_frozenset(fz_string):
    try:
        cleaned = fz_string.replace("frozenset(", "").rstrip(")")
        return set(ast.literal_eval(cleaned))
    except:
        return set()

df['parsed_antecedents'] = df['antecedents'].astype(str).apply(parse_frozenset)


In [ ]:

# Encode
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(df['parsed_antecedents'])
y = df['label'].values


In [ ]:

# 10-fold CV
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_aucs = []
models = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    model = XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        max_depth=4,
        n_estimators=60,
        learning_rate=0.1,
        verbosity=0
    )
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, preds)
    print(f"Fold {fold} AUC: {auc:.3f}")
    
    cv_aucs.append(auc)
    models.append(model)

print(f"Mean AUC across folds: {np.mean(cv_aucs):.3f}")
